# Ophthalmic Dataset EDA
Exploratory data analysis for `full_df.csv`.

In [ ]:
import sys
sys.path.insert(0, '../backend')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0a0a0a'
matplotlib.rcParams['axes.facecolor'] = '#111111'
matplotlib.rcParams['text.color'] = '#e2e8f0'
matplotlib.rcParams['axes.labelcolor'] = '#e2e8f0'
matplotlib.rcParams['xtick.color'] = '#e2e8f0'
matplotlib.rcParams['ytick.color'] = '#e2e8f0'
matplotlib.rcParams['axes.edgecolor'] = '#333'

In [ ]:
df = pd.read_csv('../data/full_df.csv', low_memory=False)
print(f'Shape: {df.shape}')
print(f'\nColumn dtypes:')
print(df.dtypes.to_string())

In [ ]:
# Null percentage per column
null_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
print('Null percentage per column (top 20):')
print(null_pct.head(20).to_string())

In [ ]:
# Plot missing values
cols_with_nulls = null_pct[null_pct > 0]
if len(cols_with_nulls) > 0:
    fig, ax = plt.subplots(figsize=(12, max(3, len(cols_with_nulls) * 0.35)))
    bars = ax.barh(cols_with_nulls.index, cols_with_nulls.values, color='#00c8ff', alpha=0.7)
    ax.set_xlabel('Missing %')
    ax.set_title('Missing Values by Column', color='#00c8ff')
    ax.grid(axis='x', alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print('No missing values!')

In [ ]:
# Column type breakdown
numerical_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() > 2]
binary_cols = [c for c in df.columns if df[c].nunique() <= 2]
categorical_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
high_card = [c for c in categorical_cols if df[c].nunique() > 50]

print(f'Numerical: {len(numerical_cols)}')
print(f'Binary:    {len(binary_cols)}')
print(f'Categorical: {len(categorical_cols)}')
print(f'High cardinality (>50): {len(high_card)}')

In [ ]:
# Target column distribution
target_keywords = ['disease', 'severity', 'grade', 'label', 'target', 'class', 'diagnosis', 'stage', 'outcome']
target_col = None
for kw in target_keywords:
    matches = [c for c in df.columns if kw in c.lower()]
    if matches:
        target_col = matches[0]
        break
if target_col is None:
    target_col = df.columns[-1]

print(f'Target column: {target_col}')
print(df[target_col].value_counts())

In [ ]:
# Target class distribution plot
vc = df[target_col].value_counts()
if len(vc) <= 30:
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ['#00c8ff' if i == 0 else '#7c3aed' if i == 1 else '#ff6b35' for i in range(len(vc))]
    ax.bar(vc.index.astype(str), vc.values, color=colors, alpha=0.8)
    ax.set_title(f'Class Distribution: {target_col}', color='#00c8ff')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    ax.grid(axis='y', alpha=0.2)
    if vc.min() > 0:
        ax.text(0.98, 0.95, f'Imbalance ratio: {vc.max()/vc.min():.2f}x',
                transform=ax.transAxes, ha='right', va='top', color='#ff6b35', fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# Numerical feature distributions
if numerical_cols:
    n_plots = min(12, len(numerical_cols))
    cols_to_plot = numerical_cols[:n_plots]
    n_rows = (n_plots + 3) // 4
    fig, axes = plt.subplots(n_rows, 4, figsize=(16, n_rows * 3))
    axes = axes.flatten()
    for i, col in enumerate(cols_to_plot):
        data = df[col].dropna()
        axes[i].hist(data, bins=30, color='#00c8ff', alpha=0.7, edgecolor='none')
        axes[i].set_title(col, fontsize=9, color='#00c8ff')
        axes[i].grid(alpha=0.15)
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle('Numerical Feature Distributions', color='#e2e8f0', fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation matrix for numerical features
if len(numerical_cols) >= 2:
    corr = df[numerical_cols[:20]].corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(corr.columns, fontsize=8)
    plt.colorbar(im)
    ax.set_title('Numerical Feature Correlation Matrix', color='#00c8ff')
    plt.tight_layout()
    plt.show()

In [ ]:
# Run full dataset pipeline for validation
from core.dataset import load_data

train_loader, val_loader, test_loader, metadata = load_data('../data/full_df.csv', batch_size=32)

print('\nFeature Metadata:')
for k, v in metadata.items():
    if k not in ('preprocessor',) and not isinstance(v, dict):
        print(f'  {k}: {v}')

In [ ]:
# Sample batch shape verification
batch = next(iter(train_loader))
print(f'X shape: {batch["X"].shape}')
print(f'M shape: {batch["M"].shape}')
print(f'y shape: {batch["y"].shape}')
print(f'y dtype: {batch["y"].dtype}')
print(f'y unique: {batch["y"].unique().tolist()}')